# PCA on B0: training and tuning, step by step

The workbench to train the PCA baseline and choose its parameters. Every number comes from a cross-validation (CV) on the healthy sessions, in `anomaly/models/cv.py`, on the protocol of `anomaly/models/evaluate.py`. Run it once from the top, then change the cells marked **edit** and rerun from there.

Steps:
1. **Folds**: the clean train and val sessions in K folds of whole sessions.
2. **One CV by hand**: set every parameter, fit, and look inside each fold: what the PCA keeps, the thresholds, who raises false alarms and why, how healthy and faulty steps separate, each dev fault.
3. **Sweep**: one parameter over a list of values, the others fixed, to see where the search should look.
4. **Optuna**: the search space (a choice or a range for any parameter), the sampler and the budget.
5. **The study**: the trials, the Pareto front of F1 and top@3, which parameters matter, and any trial opened as in step 2.
6. **Choice**: the rule fixed before the search, the comparison with the defaults, the record in `results/tuning/pca-B0/<app>/`.

How the CV works:
- **fold i**: the PCA is fit on the other folds; the threshold of each service is its largest persistent score over those fit sessions; then the model scores the sessions of fold i (healthy: every alarm there is false) and every dev fault (the PCA never trains on a fault, so every fold scores them all);
- **pooled**: every inference counts once. False alarms are summed over the folds (each healthy step is held out once); fault inferences are averaged over the K models, and so are found, minutes and top@k;
- **objectives**: F1 and top@3 of the reference system (`model+restart+absence`).

The faults choose the parameters, never the weights: the PCA stays trained on healthy steps only. Test, test-unseen, noop and the test faults are never read here: they are kept for the report, read once with the final parameters.

Fixed, because they are the protocol shared with Orcrist: the warm-up (300 s), the persistence (2 steps), the threshold (largest healthy score of the fit sessions), the rules (restart, absence) and the variants.

Terms:
- **degree**: score / threshold; above 1 the service is in alarm;
- **healthy in alarm %**: the share of held-out healthy steps in which the system is in alarm (false alarms per inference);
- **found**: an alarm starts after the injection; **top@k**: a target is among the first k services ranked at that first alarm.

In [ ]:
import sys
import warnings
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from anomaly import build
from anomaly.models import cv, evaluate as ev, pca

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.precision", 3)

APP = "hotel"  # edit: "hotel" or "social"
K_FOLDS = 5  # edit: folds of the CV
DATA = build.DATASET / "built"
OUT = cv.OUT  # studies and choices go to OUT/pca-B0/<app>/
V = cv.VARIANT  # the variant the figures follow: the reference system

RUNS = ev.load(APP, DATA)
SERVICES = RUNS[0].services
FEATURES = list(pca.PCA.features)
print(f"{APP}: {len(RUNS)} runs, {len(SERVICES)} services; {sum(r.role in ('train', 'val') for r in RUNS)} clean train "
      f"and val sessions, {sum(r.role == 'fault-dev' for r in RUNS)} dev faults")

In [ ]:
# Colours: categorical slots in a fixed order (as in pca-B0); sequential blue for good counts, orange for bad ones
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, AQUA, YELLOW, RED, NEUTRAL, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948", "#f0efec", "#b9b7b1"
DIVERGING = LinearSegmentedColormap.from_list("ratio", [BLUE, NEUTRAL, RED])
SEQUENTIAL = LinearSegmentedColormap.from_list("seq", ["#f2f7fd", "#cde2fb", "#2a78d6", "#104281"])
SEQUENTIAL_BAD = LinearSegmentedColormap.from_list("bad", ["#fdf4ef", "#f9cdb8", "#eb6834", "#8a3110"])  # darker = worse
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": .6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False, "lines.linewidth": 2,
    "figure.dpi": 110})


def rep(r):
    """The repetition of a run: r13."""
    return r.id.split("__")[-1]


def batch(r):
    return "r1-r12" if int(rep(r)[1:]) <= 12 else "r13+"


def heat(ax, M, cmap, labels, fmt, vmin=None, vmax=None, dark=.6):
    """A heatmap with every cell written; NaN cells left blank."""
    ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    lo, hi = np.nanmin(M) if vmin is None else vmin, np.nanmax(M) if vmax is None else vmax
    for (i, j), x in np.ndenumerate(M):
        if np.isfinite(x):
            ax.text(j, i, fmt(x), ha="center", va="center", fontsize=8,
                    color="white" if hi > lo and (x - lo) / (hi - lo) > dark else INK)
    ax.set_yticks(range(len(labels)), labels)
    ax.grid(False)

## 1. Folds

The clean train and val sessions, whole (steps next to each other are near copies: a random split would put the twin of every held-out step in the fit), in K folds balanced by loaded steps. Each fold is held out once: the fit uses the other K − 1, about (K − 1)/K of the steps the final model trains on. The two batches of sessions (r1-r12 and r13+, collected at different times) are mixed in every fold.

In [ ]:
FOLDS = cv.folds(RUNS, K_FOLDS)
steps = lambda f: sum(int(r.on.sum()) for r in f)
display(pd.DataFrame([{"fold": i, "sessions": " ".join(rep(r) + (" (val)" if r.role == "val" else "") for r in f),
                       "steps": steps(f), "hours": round(steps(f) * 30 / 3600, 1),
                       "r1-r12": sum(batch(r) == "r1-r12" for r in f), "r13+": sum(batch(r) == "r13+" for r in f)}
                      for i, f in enumerate(FOLDS)]).set_index("fold"))

fig, ax = plt.subplots(figsize=(10, .5 * K_FOLDS + 1.4))
for i, f in enumerate(FOLDS):
    left = 0
    for r in f:
        n = int(r.on.sum())
        ax.barh(i, n - 4, left=left + 2, color=BLUE if batch(r) == "r1-r12" else AQUA, height=.7)
        ax.text(left + n / 2, i, rep(r), ha="center", va="center", fontsize=8, color="white")
        left += n
ax.set_yticks(range(K_FOLDS), [f"fold {i}" for i in range(K_FOLDS)])
ax.invert_yaxis()
ax.set(xlabel="healthy loaded steps (30 s)", title=f"{APP}: the sessions of each fold")
ax.legend(handles=[Patch(color=BLUE, label="r1-r12"), Patch(color=AQUA, label="r13+")], loc="lower left",
          bbox_to_anchor=(0, 1.02), ncol=2)
ax.grid(axis="y", visible=False)
plt.show()

## 2. One cross-validation by hand

**edit** `HP`: every parameter of the PCA (the defaults are `pca.HP`, those of `results/pca-B0`):
- `VAR`: the share of the healthy variance the subspace keeps. Lower keeps fewer directions: more of a healthy step falls in the residual (more false alarms), but a fault has fewer directions to hide in;
- `SD_FLOOR`: the floor of the deviation that z-scores a feature, in prepare units (log1p; memory as its change per step). A feature almost flat in training (the threads of an idle mongo) has a tiny deviation, so any change explodes its z-score: the floor caps that. 0 is off;
- `MIN_FINITE`: a feature enters the model of a service when it is finite in this share of the fit steps;
- `MIN_SD`: a feature flatter than this in the fit steps is left out.

The next cell defines the tables and figures of this section; section 5 reuses them to open any trial.

In [ ]:
def tables(res):
    """Per fold (the reference system) and pooled (every variant)."""
    display(res.table[res.table.variant == V].drop(columns="variant").set_index("fold").round(3))
    display(res.pooled.T.round(3))


def fig_model(res, fold=0):
    """What the PCA keeps: the cumulative variance per service (one fold), the directions kept in each fold, and how
    the threshold of each service moves between folds."""
    f = res.folds[fold]
    n = len(FEATURES)
    cum = np.array([np.r_[m.cumulative, [np.nan] * (n - len(m.cumulative))] if m else [np.nan] * n
                    for m in f.model.services])
    k = np.array([[m.V.shape[1] if m else np.nan for m in x.model.services] for x in res.folds]).T
    thr = np.array([x.thr for x in res.folds]).T
    ratio = thr / np.nanmedian(np.where(np.isfinite(thr), thr, np.nan), 1, keepdims=True)
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, .32 * len(SERVICES) + 2.2), sharey=True,
                                     gridspec_kw={"width_ratios": [n, K_FOLDS, K_FOLDS]})
    heat(a1, 100 * cum, SEQUENTIAL, SERVICES, lambda x: f"{x:.0f}", 0, 100)
    for i, m in enumerate(f.model.services):
        if m:
            a1.add_patch(plt.Rectangle((m.V.shape[1] - 1.5, i - .5), 1, 1, fill=False, ec=ORANGE, lw=2))
    a1.set_xticks(range(n), range(1, n + 1))
    a1.set(xlabel="directions (a row has one cell per feature in the model)",
           title=f"cumulative variance %, fold {fold}\n(orange: k, the first to keep VAR)")
    heat(a2, k, SEQUENTIAL, SERVICES, lambda x: f"{x:.0f}")
    a2.set_xticks(range(K_FOLDS))
    a2.set(xlabel="fold", title="k per fold")
    heat(a3, np.log2(ratio), DIVERGING, SERVICES, lambda x: f"{2 ** x:.2f}", -1, 1, dark=2)
    a3.set_xticks(range(K_FOLDS))
    a3.set(xlabel="fold", title="threshold / its median\nover the folds")
    fig.tight_layout()
    plt.show()


def false_alarms(res):
    """Per held-out healthy step and service in alarm: the fold, the service, the features that make the score (share)."""
    rows = []
    for f in res.folds:
        for r in f.held:
            for t, i in zip(*np.where(r.alarm[V] & r.on[:, None])):
                c = r.r2[t, i]
                rows.append({"fold": f.i, "session": rep(r), "service": SERVICES[i], "degree": r.norm[t, i],
                             "rule": bool(r.rule[V][t, i]), **dict(zip(FEATURES, c / c.sum() if c.sum() else c))})
    return pd.DataFrame(rows, columns=["fold", "session", "service", "degree", "rule", *FEATURES])


def fig_false_alarms(res):
    fa = false_alarms(res)
    count = fa.groupby(["service", "fold"]).size().unstack().reindex(index=SERVICES, columns=range(K_FOLDS)).fillna(0)
    share = fa.groupby("service")[FEATURES].mean().reindex(SERVICES)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, .32 * len(SERVICES) + 2.4), sharey=True,
                                 gridspec_kw={"width_ratios": [K_FOLDS + 1, len(FEATURES)]})
    heat(a1, np.c_[count.values, count.sum(axis=1).values], SEQUENTIAL_BAD, SERVICES, lambda x: f"{x:.0f}" if x else "")
    a1.set_xticks(range(K_FOLDS + 1), [*range(K_FOLDS), "all"])
    a1.set(xlabel="fold", title="held-out healthy steps in alarm")
    heat(a2, 100 * share.values, SEQUENTIAL_BAD, SERVICES, lambda x: f"{x:.0f}", 0, 100)
    a2.set_xticks(range(len(FEATURES)), FEATURES, rotation=40, ha="right")
    a2.set_title("what makes those alarms: mean share of the score %")
    fig.suptitle(f"false alarms: {len(fa)} service-steps, {fa.rule.sum()} by a rule", fontweight="bold")
    fig.tight_layout()
    plt.show()


def fig_separation(res):
    """ECDF of the system degree (the largest over the services) on held-out healthy steps and on dev fault steps."""
    h = np.concatenate([r.norm[r.on].max(1) for f in res.folds for r in f.held])
    x = np.concatenate([r.norm[r.fault].max(1) for f in res.folds for r in f.dev])
    fig, ax = plt.subplots(figsize=(9, 4))
    for d, c, name in ((h, BLUE, "held-out healthy steps"), (x, ORANGE, "dev fault steps")):
        d = np.sort(np.clip(d, 1e-2, None))
        ax.step(d, np.arange(1, len(d) + 1) / len(d), where="post", color=c,
                label=f"{name}: {100 * (d > 1).mean():.1f}% above 1")
    ax.axvline(1, color=INK2, ls=":", lw=1)
    ax.set_xscale("log")
    ax.set(xlabel="system degree (largest score / threshold over the services; model only, no rule)",
           ylabel="share of steps at or below", title="separation of healthy and faulty steps, all folds")
    ax.legend(loc="upper left")
    fig.tight_layout()
    plt.show()


def first_rank(r):
    """Rank of the best ranked target at the first alarm after the injection (1 = first), NaN when not found."""
    first = ev.starts(r.alarm[V]).any(1) & r.fault
    if not first.any():
        return np.nan
    k = int(first.argmax())
    order = list(ev.rank(r.norm[k], r.rule[V][k]))
    return min(order.index(t) for t in r.targets) + 1


def fig_faults(res):
    """Each dev fault in each fold: the rank of its target at the first alarm; blank = not found."""
    dev = [r for r in res.folds[0].dev if r.targets]
    R = np.array([[first_rank(r) for r in f.dev if r.targets] for f in res.folds]).T
    fig, ax = plt.subplots(figsize=(7, .27 * len(dev) + 1.8))
    heat(ax, np.minimum(R, 6), SEQUENTIAL.reversed(), [f"{r.scenario} {rep(r)}" for r in dev],
         lambda x: "", 1, 6)
    for (i, j), x in np.ndenumerate(R):
        ax.text(j, i, "miss" if np.isnan(x) else f"{x:.0f}", ha="center", va="center", fontsize=8,
                color="white" if x <= 2 else INK)
    ax.set_xticks(range(K_FOLDS))
    ax.set(xlabel="fold", title="rank of the target at the first alarm\n(1-3: top@3; miss: not found)")
    fig.tight_layout()
    plt.show()

In [ ]:
HP = {  # edit
    "VAR": 0.95,
    "SD_FLOOR": 0.0,
    "MIN_FINITE": 0.95,
    "MIN_SD": 1e-6,
}
res = cv.cross_validate("pca", RUNS, HP, K_FOLDS)
tables(res)

### 2a. What the PCA keeps, and how stable it is

Left: per service, the variance kept by the first directions (one fold; the folds differ little), and k, the first count to reach `VAR`. Middle: k in every fold. Right: the threshold of each service in each fold, over its median: a threshold that moves a lot between folds depends on which sessions were fit, a sign that the healthy behaviour of that service is not well covered.

In [ ]:
fig_model(res)

### 2b. False alarms: who and why

Every held-out healthy step where a service is in alarm. Left: how many per service and fold. Right: the features that make their score. A false alarm concentrated on one feature of one service is the case a floor on the deviation (`SD_FLOOR`) or a different `VAR` can fix; one spread over folds and features is the normal variability of the service.

In [ ]:
fig_false_alarms(res)
false_alarms(res).sort_values("degree", ascending=False).head(15).round(2)

### 2c. Separation and each fault

The system degree on held-out healthy steps against dev fault steps: the threshold is the line at 1. Then every dev fault in every fold: a fault that moves between found and missed, or between ranks, across folds sits near the threshold.

In [ ]:
fig_separation(res)
fig_faults(res)

## 3. Sweep: one parameter at a time

**edit** `SWEEP`: a parameter of `HP` and the values to try; the others stay as in `HP`. One CV per value. The thick line is pooled; the dots are the folds (F1 is pooled only); the dashed line is the value in `HP`. Use it to choose the ranges of the search in section 4: where the curves are flat the parameter matters little, where they bend is the region to search.

In [ ]:
SWEEP = ("VAR", [0.80, 0.85, 0.90, 0.95, 0.97, 0.99])  # edit
param, values = SWEEP
rows, fold_rows = [], []
for x in values:
    r_ = cv.cross_validate("pca", RUNS, HP | {param: x}, K_FOLDS)
    rows.append({param: x, **r_.pooled.loc[V].to_dict()})
    fold_rows += [{param: x, **row} for row in r_.table[r_.table.variant == V].to_dict("records")]
sweep, sweep_folds = pd.DataFrame(rows), pd.DataFrame(fold_rows)

METRICS = ["F1", "top@3 %", "healthy in alarm %", "recall", "faults found %", "top@1 %"]
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True)
log = min(values) > 0 and max(values) / min(values) >= 20
for ax, m in zip(axes.flat, METRICS):
    if m in sweep_folds:
        ax.scatter(sweep_folds[param], sweep_folds[m], s=14, color=MUTED, zorder=2)
    ax.plot(sweep[param], sweep[m], color=BLUE, marker="o", ms=5, zorder=3)
    ax.axvline(HP[param], color=INK2, ls="--", lw=1)
    ax.set_title(m)
    if log:
        ax.set_xscale("log")
for ax in axes[1]:
    ax.set_xlabel(param)
fig.suptitle(f"{APP}: sweep of {param} (the rest as in HP)", fontweight="bold")
fig.tight_layout()
plt.show()
sweep.set_index(param)[["F1", "precision", "recall", "healthy in alarm %", "faults found %", "top@1 %", "top@3 %",
                        "top@5 %", "minutes to alarm, median", "healthy in alarm % sd", "top@3 % sd"]].round(3)

## 4. Optuna

**edit** `SPACE`: every parameter of the PCA, each one
- a list, a choice among values: `[0.80, 0.90, 0.95]`;
- `(lo, hi)`, a range (of integers when both are integers): `(0.0, 0.3)`;
- `(lo, hi, "log")`, a range in log scale: `(1e-3, 1e-1, "log")`;
- any other value, fixed: `0.95`.

`SAMPLER`: `"grid"` tries every combination of the lists (no range allowed) and stops by itself; `"tpe"` (Bayesian, multi-objective) or `"random"` run until the study has `N_TRIALS` complete trials, or for `TIMEOUT_H` hours. A tpe or random study starts with the defaults, the reference, when they are in the space. (Long searches can also run in background processes, as in `train-orcrist-A.ipynb`; the PCA rarely needs it.)

`STUDY`: the name of the study. It is kept in `OUT/pca-B0/<app>/study.journal` (not in git): rerunning the cell resumes it (a grid skips what it has done). A study refuses another space, sampler, folds or dev set: a new space is a new name, so the trials of a study are always comparable.

Each trial is one CV as in section 2 (about a second for the PCA); a line per trial. No trial starts while the nightly collection runs or just before it (17:45-18:05 UTC): the cell waits.

In [ ]:
SPACE = {  # edit
    "VAR": [0.80, 0.85, 0.90, 0.95, 0.99],
    "SD_FLOOR": [0.0, 0.05, 0.1, 0.2],
    "MIN_FINITE": 0.95,
    "MIN_SD": 1e-6,
}
SAMPLER = "grid"  # edit: "grid", "tpe" or "random"
STUDY = f"pca-{APP}-1"  # edit: a new name for a new space
N_TRIALS = None  # edit: tpe and random need N_TRIALS or TIMEOUT_H
TIMEOUT_H = None

study = cv.study("pca", APP, RUNS, SPACE, STUDY, SAMPLER, K_FOLDS, out=OUT)
cv.optimize(study, "pca", RUNS, SPACE, N_TRIALS, TIMEOUT_H, K_FOLDS)

## 5. The study

All the complete trials, best F1 first. `pareto`: no other trial is better on both F1 and top@3. `reference`: the default parameters. The `sd` columns are the deviation across folds: two trials whose difference is within it, or within one dev fault (about 4 points of top@3 or found), are not distinguishable.

In [ ]:
df = cv.trials(study)
PARAMS = [p for p, s in SPACE.items() if isinstance(s, (list, tuple))]
COLS = PARAMS + ["F1", "precision", "recall", "healthy in alarm %", "faults found %", "top@1 %", "top@3 %", "top@5 %",
                 "minutes to alarm, median", "healthy in alarm % sd", "top@3 % sd", "pareto", "reference"]
df[COLS].sort_values("F1", ascending=False).round(3).head(25)

In [ ]:
chosen = cv.choose(df)
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.2, 1, 1]})
a1.scatter(df.F1, df["top@3 %"], s=22, color=MUTED, label="trial")
front = df[df.pareto].sort_values("F1")
a1.plot(front.F1, front["top@3 %"], color=BLUE, marker="o", ms=6, lw=1.2, label="Pareto front")
ref = df[df.reference]
a1.scatter(ref.F1, ref["top@3 %"], s=110, marker="*", color=ORANGE, zorder=4, label="reference (defaults)")
a1.scatter([chosen.F1], [chosen["top@3 %"]], s=140, facecolor="none", edgecolor=INK, lw=1.6, zorder=5,
           label=f"chosen (trial {chosen.name})")
a1.set(xlabel="F1", ylabel="top@3 %", title="F1 against top@3")
a1.legend(loc="upper left", fontsize=8)
for ax, m in ((a2, "F1"), (a3, "top@3 %")):
    ax.scatter(df.index, df[m], s=14, color=MUTED)
    ax.plot(df.index, df[m].cummax(), color=BLUE, lw=1.5, label="best so far")
    ax.set(xlabel="trial", title=f"{m} by trial")
    ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

Which parameters matter (PED-ANOVA: how much each parameter's values differ between the best trials and the rest, per objective), and every trial against every parameter. With few trials or one parameter the importance is not computed.

In [ ]:
from optuna.importance import PedAnovaImportanceEvaluator, get_param_importances

warnings.filterwarnings("ignore", message="PedAnovaImportanceEvaluator computes")  # handled: the target is negated

if len(df) >= 10 and len(PARAMS) >= 2:
    imp = pd.DataFrame({o: get_param_importances(study, evaluator=PedAnovaImportanceEvaluator(), params=PARAMS,
                                                 target=lambda t, j=j: -t.values[j])  # it ranks low values first
                        for j, o in enumerate(cv.OBJECTIVES)}).reindex(PARAMS)
    display(imp.round(3))
fig, axes = plt.subplots(2, len(PARAMS), figsize=(4 * len(PARAMS) + 1, 6), sharey="row", squeeze=False)
for j, p in enumerate(PARAMS):
    for i, m in enumerate(cv.OBJECTIVES):
        ax = axes[i, j]
        ax.scatter(df[p], df[m], s=18, c=np.where(df.pareto, BLUE, MUTED))
        if isinstance(SPACE[p], tuple) and SPACE[p][2:] == ("log",):
            ax.set_xscale("log")
        ax.set(xlabel=p if i else None, ylabel=m if not j else None)
fig.legend(handles=[Line2D([], [], ls="", marker="o", color=BLUE, label="Pareto"),
                    Line2D([], [], ls="", marker="o", color=MUTED, label="other")], loc="upper right", ncol=2)
fig.suptitle("every trial against every parameter", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, .95))
plt.show()

### 5a. Open a trial

**edit** `TRIAL`: a trial number of the table (None: the one the rule of section 6 chooses). Its CV is rerun with its parameters and shown as in section 2.

In [ ]:
TRIAL = None  # edit
t = int(chosen.name) if TRIAL is None else TRIAL
hp = study.trials[t].user_attrs["hp"]
print(f"trial {t}: {hp}")
res_t = cv.cross_validate("pca", RUNS, hp, K_FOLDS)
tables(res_t)
fig_model(res_t)
fig_false_alarms(res_t)
fig_separation(res_t)
fig_faults(res_t)

## 6. Choice

The rule, fixed before looking at the trials: on the Pareto front, the largest F1 among the trials whose top@3 is within `cv.TOLERANCE` points (about one dev fault) of the best. Against the reference: a gain within the deviation across folds, or within one or two dev faults, is noise, and the defaults are as good.

**edit** `SAVE = True` writes `trials.csv` and `choice.json` (chosen and reference trial, the rule, the folds, the data, the code) to `OUT/pca-B0/<app>/`. These are the record of the study; the final parameters are frozen only on the complete dataset, and the test is read once, with them.

In [ ]:
SAVE = False  # edit
compare = pd.concat([df[df.reference].assign(row="reference (defaults)"),
                     df.loc[[chosen.name]].assign(row=f"chosen (trial {chosen.name})")]).set_index("row")[COLS]
display(compare.T)
if SAVE:
    print("written to", cv.save(study, "pca", APP, chosen, OUT))